In [1]:
!pip install transformers torch scikit-learn pandas nltk

In [3]:
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from transformers import DistilBertTokenizer, DistilBertModel
from torch.utils.data import Dataset, DataLoader

In [11]:
# load the dataset
df = pd.read_csv("../data/synthetic_grievance_dataset.csv")

print(df.head())
print(df.shape)

                                           complaint      category  \
0  Kindly look into the matter: received damaged ...  Consumer Law   
1  This problem occurred because received damaged...  Consumer Law   
2  This problem occurred because seller refused t...  Consumer Law   
3  I am writing to report that wrong product deli...  Consumer Law   
4            Received damaged item from online store  Consumer Law   

         date     region  
0  2024-02-29      Delhi  
1  2023-05-06  Hyderabad  
2  2022-08-13    Kolkata  
3  2023-09-28    Chennai  
4  2024-05-08  Ahmedabad  
(2000, 4)


In [15]:
# label encoding
le = LabelEncoder()
df['label'] = le.fit_transform(df['category'])

print(le.classes_)

['Consumer Law' 'Cyber Crime' 'Domestic Violence' 'Property Dispute']


In [17]:
# train - test split
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df['complaint'],
    df['label'],
    test_size=0.2,
    random_state=42
)

In [19]:
# tokenization
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

train_encodings = tokenizer(
    list(train_texts),
    truncation=True,
    padding=True,
    max_length=128
)

val_encodings = tokenizer(
    list(val_texts),
    truncation=True,
    padding=True,
    max_length=128
)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

In [34]:
class GrievanceDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels.iloc[idx], dtype=torch.long)
        return item

    def __len__(self):
        return len(self.labels)

In [36]:
# dataloaders
train_dataset = GrievanceDataset(train_encodings, train_labels)
val_dataset = GrievanceDataset(val_encodings, val_labels)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8)

In [38]:
# model definition
class DistilBERTClassifier(nn.Module):
    def __init__(self, num_classes):
        super(DistilBERTClassifier, self).__init__()
        self.bert = DistilBertModel.from_pretrained('distilbert-base-uncased')
        self.fc = nn.Linear(768, num_classes)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        hidden_state = outputs.last_hidden_state[:, 0]  # CLS token
        return self.fc(hidden_state)

In [40]:
# initialize a model
device = torch.device("cpu")  # CPU for your laptop

model = DistilBERTClassifier(num_classes=len(le.classes_))
model.to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=2e-5)
loss_fn = nn.CrossEntropyLoss()

In [42]:
# train
epochs = 3

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for batch in train_loader:
        optimizer.zero_grad()

        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids, attention_mask)

        loss = loss_fn(outputs, labels)
        total_loss += loss.item()

        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader)}")

Epoch 1, Loss: 0.15831883663195184
Epoch 2, Loss: 0.001958549187402241
Epoch 3, Loss: 0.0009294754348229617


In [44]:
# evaluation
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for batch in val_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids, attention_mask)
        _, predicted = torch.max(outputs, dim=1)

        correct += (predicted == labels).sum().item()
        total += labels.size(0)

accuracy = correct / total
print("Validation Accuracy:", accuracy)

Validation Accuracy: 1.0


In [52]:
# save model
torch.save(model.state_dict(), "save_model/model.pt")